# R32 — Run all dari Release R4 publik

Pilih runtime CPU lalu **Runtime → Run all**. Arsip publik diunduh otomatis dan diperiksa checksum; tidak membutuhkan token, Drive, unggahan ZIP atau GPU. Ini pengujian analisis tiga notebook dari prediksi tersimpan, bukan training ulang. Simpan ZIP keluaran dan screenshot sel hasil untuk bukti. Notebook yang belum dijalankan tidak membuktikan kelulusan Colab.

In [ ]:
import hashlib, json, os, subprocess, sys, tempfile, urllib.request, zipfile
from pathlib import Path
import google.colab
from google.colab import files
assert os.environ.get('COLAB_RELEASE_TAG') or Path('/opt/google/colab').exists(), 'Gunakan runtime Colab asli'
directory = Path(tempfile.mkdtemp(prefix='tempo_public_r32_'))
archive = directory/'R4.zip'
urllib.request.urlretrieve('https://github.com/yusepahmad/tempo-clip-indobert-xai/releases/download/v2026.11.08-r4/TEMPO_REVISI_II_R4_PUBLIC_20261108.zip', archive)
digest = hashlib.sha256()
with archive.open('rb') as stream:
    for chunk in iter(lambda: stream.read(4*1024*1024), b''):
        digest.update(chunk)
assert digest.hexdigest() == 'a2c9a67f7ee02472f76a0581b364dfc96b6c7bd6584cb71a3cac5a090a8d2035', 'Checksum arsip berbeda'
with zipfile.ZipFile(archive) as bundle:
    for name in bundle.namelist():
        assert (directory/name).resolve().is_relative_to(directory), 'Path ZIP tidak aman'
    bundle.extractall(directory)
ROOT = directory/'joy_revisi_ii'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT/'requirements-analysis-lock.txt')], check=True)

In [ ]:
output = directory/'hasil_colab'
subprocess.run([sys.executable, str(ROOT/'tools/run_clean_notebooks.py'), '--root', str(ROOT), '--output', str(output)], check=True)
result = json.loads((output/'RUNTIME_VERIFICATION.json').read_text())
assert result['status'] == 'verified' and result['colab_verified'] is True
assert len(result['notebooks']) == 3 and all(r['errors'] == 0 for r in result['notebooks'])
print('R32 — Run all Colab LULUS; ambil screenshot sel ini')
print(json.dumps(result, indent=2))

In [ ]:
import shutil
evidence = shutil.make_archive(str(directory/'R32_BUKTI_COLAB'), 'zip', output)
files.download(evidence)